# Example 7 — XSL DR3 reference-star validation

## What this example teaches

This example shows how Spyctres treats XSL DR3 as a product-specific reader and validation data set. XSL spectra were observed with X-SHOOTER, but the DR3 files are merged library products: the reader preserves their arm/overlap resolution provenance, air-wavelength convention, and stellar-rest-frame status instead of treating them like generic reduced X-SHOOTER arms.

## Requirements

The first inspection cells only need the bundled XSL DR3 sample files and the saved coarse validation JSON in `examples/data/`. A fresh validation run requires a configured local PHOENIX library.

## Expected outputs

You should see a reader/product summary, a compact table comparing saved Spyctres fits with literature reference parameters, and one observed/model validation panel. The panel is diagnostic: ordinary targets and stress/unsupported targets should not be mixed into one accuracy claim.


## 0. Imports and controls

We use the public one-import path. The reader name describes the data product (`xsl_dr3`), not just the telescope/instrument that originally observed it. The path setup below is deliberately independent of the Jupyter working directory, so the notebook works whether you launch Jupyter from the repository root or from `examples/`.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import Spyctres as sp

# Resolve paths from the imported Spyctres source tree
ROOT = Path(sp.__file__).resolve().parents[1]
MANIFEST = ROOT / "examples" / "xsl_validation_manifest.csv"
RESULTS = Path(sp.example_data_path("xsl_figure1_validation_coarse_results.json"))
EXAMPLE_TARGET = Path(sp.example_data_path("xsl_spectrum_X0245_merged.fits"))

RUN_VALIDATION = True
PLOT_SCALE = "global"


## 1. Inspect the XSL reader and one bundled target

`sp.list_readers()` lists every reader Spyctres knows about. `sp.get_reader_info("xsl_dr3")` records the assumptions for this particular product. XSL DR3 released spectra are already air-wavelength, stellar-rest products, so Spyctres records the observer-motion frame as not applicable and treats fitted RV as a residual alignment check, not a new physical stellar RV. If your spectrum is not covered by an existing reader, the safest path is to write a small reader that returns a `SpectrumSegment` or `SpectrumCollection` with explicit wavelength, uncertainty, mask, frame, and resolution metadata.


In [ ]:
print("Available readers:", ", ".join(sp.list_readers()))
reader_info = sp.get_reader_info("xsl_dr3").to_metadata()
for key in [
    "canonical_name",
    "expected_file_type",
    "wavelength_unit",
    "default_wave_medium",
    "default_observer_frame",
    "default_stellar_rest_status",
    "segment_structure",
    "resolving_power",
    "notes",
]:
    print(f"{key}: {reader_info.get(key)}")

xsl_spec = sp.read_spectrum(EXAMPLE_TARGET, reader="xsl_dr3")
print("One bundled XSL target:")
print(xsl_spec.summary())
print("stellar_rest:", xsl_spec.stellar_rest)
print("wavelength_medium:", xsl_spec.wavelength_medium)
print("needs_barycentric_correction:", xsl_spec.needs_barycentric_correction)
print("fitted_rv_kms_role:", xsl_spec.meta.get("fitted_rv_kms_role"))


## 2. Inspect the saved validation payload

The saved payload lets a new user see the structure of the validation output without rebuilding the PHOENIX cache. Reference parameters are used only for comparison after the fit; they are not fit priors.


In [ ]:
payload = json.loads(Path(RESULTS).read_text(encoding="utf-8"))
rows = payload["results"]

status_counts = {}
for row in rows:
    status_counts[row.get("status", "unknown")] = status_counts.get(row.get("status", "unknown"), 0) + 1

print("Targets:", len(rows))
print("Status counts:", status_counts)
print("Wave-medium assumption:", payload.get("wave_medium_assumption"))
print("Fit range [Å]:", payload.get("fit_wave_range_A"))

print("ID      role                 status        fit Teff  ref Teff")
print("----------------------------------------------------------------")
for row in rows:
    fit = row.get("fit") or {}
    ref = row.get("reference") or {}
    fit_teff = "—" if fit.get("teff") is None else f"{float(fit['teff']):.0f}"
    ref_teff = "—" if ref.get("teff") is None else f"{float(ref['teff']):.0f}"
    print(
        f"{row.get('xsl_id', ''):<7} {row.get('validation_role', ''):<20} "
        f"{row.get('status', ''):<13} {fit_teff:>8} {ref_teff:>8}"
    )


## 3. Plot one observed/model validation panel

The default display scale is global per star. This avoids artificial arm-to-arm jumps created by separately normalizing UVB, VIS, and NIR for display. Use `PLOT_SCALE = "per_segment"` only for line-shape diagnostics.


In [ ]:
example_row = next(row for row in rows if row.get("validation_plot"))
fig, axes = sp.plot_xsl_validation_payload(
    example_row["validation_plot"],
    scale_mode=PLOT_SCALE,
    title=(
        f"{example_row.get('xsl_id')}: {example_row.get('spectral_type')} "
        f"[{example_row.get('validation_role')}]"
    ),
)
plt.show()


## 4. Optional: rerun the validation

Set `RUN_VALIDATION = True` only when PHOENIX is configured. The runner is resumable and writes after each target, so it is suitable for a longer validation pass. Stress and unsupported targets are reported separately from ordinary recovery statistics.


In [ ]:
output_json = Path("/tmp/spyctres_example7_xsl_results.json")
command = [
    sys.executable,
    str(ROOT / "scripts" / "xsl_validation.py"),
    str(MANIFEST),
    "--output",
    str(output_json),
    "--resume",
]

if RUN_VALIDATION:
    subprocess.run(command, check=True)
    RESULTS = output_json
    print("Fresh validation written to:", RESULTS)
else:
    print("Fresh validation is disabled. To run it later:")
    print(" ".join(command))


## Interpretation checklist

- Treat `standard` targets as the ordinary recovery sample.
- Treat carbon, cool/peculiar, and very hot targets as stress tests of the current PHOENIX workflow.
- A target above the supported PHOENIX temperature range should be reported as unsupported rather than extrapolated.
- Do not apply extra arm scaling or RV correction to XSL DR3 by default; the product has already been merged and placed on its documented wavelength scale.
